# 09 — Net Revenue Retention (NRR)

Every retention metric elsewhere in this project (cohort retention in notebook 02,
Kaplan-Meier survival curves in notebook 06) measures **customer-count** retention:
what percentage of customers are still active. That's a real and useful number, but
it can hide as much as it reveals for a subscription business, because it treats
every customer as equally important regardless of how much revenue they represent,
and it can't distinguish two very different situations:

- A cohort where most customers stay, but the ones who stay quietly downgrade
- A cohort where some customers leave, but the ones who stay expand enough to
  more than make up for it

Customer-count retention reports both of these as "retention was fine" or "retention
was bad" purely based on headcount. **Net Revenue Retention (NRR)** answers a
different, often more decision-relevant question: of the *revenue* a cohort started
with, how much do we still have N months later, decomposed into exactly where it
went — expansion, contraction, or churn.

This requires longitudinal, per-customer-per-month revenue data, which the original
cross-sectional snapshot table can't provide (see `data/generate_monthly_panel.py`'s
docstring for why). The monthly revenue panel built for this notebook simulates
realistic expansion/contraction events on top of the existing churn/tenure data, so
NRR becomes computable and demonstrable on top of the same underlying customer base
used throughout the rest of the project.


In [1]:
import sys
sys.path.insert(0, '../')

import pandas as pd
from src.preprocessing import load_raw
from src.cohort import (
    compute_nrr_by_cohort_month, nrr_by_segment, overall_nrr_summary,
    plot_nrr_trend, plot_nrr_waterfall
)

customers = load_raw()
panel = pd.read_csv('../data/raw/monthly_revenue_panel.csv')
print(f'{len(panel):,} monthly billing events across {panel["customerID"].nunique():,} customers')


202,085 monthly billing events across 7,043 customers


## NRR over time

`month` here is customer-relative (month 1 = each customer's own first billed month),
which keeps this on the same timeline convention as the rest of the project's cohort
work, and lets a mixed-signup-date customer base be compared on a single axis.


In [2]:
nrr_df = compute_nrr_by_cohort_month(panel)
fig = plot_nrr_trend(nrr_df)
fig.show()

nrr_df.head(15)


,month,starting_mrr,expansion_mrr,contraction_mrr,churned_mrr,ending_mrr,nrr
0,1,471994.68,0.00,0.00,0.00,471994.68,1.000000
1,2,471994.68,3919.54,2983.07,7448.72,465482.43,0.986203
2,3,465482.43,3933.40,2857.31,7172.80,459385.72,0.986902
3,4,459385.72,3860.28,2756.16,7637.34,452852.50,0.985778
4,5,452852.50,3980.56,3077.82,7365.16,446390.08,0.985730
5,6,446390.08,3762.98,3359.24,7136.15,439657.67,0.984918
6,7,439657.67,3494.15,3035.99,7730.52,432385.31,0.983459
7,8,432385.31,3549.78,2960.42,6698.14,426276.53,0.985872
8,9,426276.53,3162.68,3037.56,8585.64,417816.01,0.980153
9,10,417816.01,3719.67,2525.54,7060.73,411949.41,0.985959


## What the trend shows

NRR starts near 100% in the earliest months (nearly everyone is still active and
un-expanded right after signup) and declines over the panel's timeline, dropping into
the 80s by months 60+. That decline is dominated by churned MRR — expansion and
contraction are both real in the simulation but small relative to the effect of
customers leaving entirely. This is a useful, honest finding on its own: it says the
revenue base's health is overwhelmingly a **retention** problem, not an
expansion/contraction problem — worth stating plainly rather than burying under an
aggregate number, since the two would call for very different business responses
(retention investment vs. upsell motion investment).


In [3]:
summary = overall_nrr_summary(nrr_df, window=12)
for k, v in summary.items():
    if isinstance(v, float):
        print(f'{k}: {v:.1%}' if 'nrr' in k else f'{k}: ${v:,.0f}')
    else:
        print(f'{k}: {v}')


avg_nrr: 81.1%
median_nrr: 83.6%
total_expansion_mrr: $2,160
total_contraction_mrr: $1,561
total_churned_mrr: $36,935
months_in_window: 12


## NRR by segment

Breaking NRR out by contract type shows whether the overall trend is uniform across
segments or concentrated in one — the same segmentation instinct used throughout this
project (and in the credit-risk project spec's segment-level diagnostic framework).


In [4]:
seg_nrr = nrr_by_segment(panel, customers, segment_col='Contract')

for seg in seg_nrr['Contract'].unique():
    s = overall_nrr_summary(seg_nrr[seg_nrr['Contract'] == seg], window=12)
    print(f'{seg:20s} avg NRR (trailing 12mo): {s["avg_nrr"]:.1%}' if s['avg_nrr'] == s['avg_nrr'] else f'{seg}: insufficient data')


Month-to-month       avg NRR (trailing 12mo): 80.8%
One year             avg NRR (trailing 12mo): 82.3%
Two year             avg NRR (trailing 12mo): 81.6%


## MRR bridge for a single month

A waterfall view of one month's NRR components makes the decomposition concrete —
this is the same "show the actual before/after mechanism, not just a summary number"
instinct that made the CLV bug-fix story land in notebook 06.


In [5]:
# pick a mid-panel month with meaningful volume
mid_month = int(nrr_df.dropna(subset=['nrr'])['month'].median())
fig = plot_nrr_waterfall(nrr_df, month=mid_month)
fig.show()


## Takeaway

NRR gives a strictly more informative view of revenue health than customer-count
retention alone: it confirms that churn, not downgrade behavior, is the dominant
force eroding the revenue base, and it does so with a decomposition that a
customer-count metric structurally cannot provide. Combined with the Cox model's
finding (notebook 07) that contract length is the single strongest independent churn
driver, this points to the same business recommendation from two independent angles:
converting month-to-month customers into longer commitments is the highest-leverage
lever for both customer retention and revenue retention.
